# Qwen 72B Dual Input Notebook

This notebook runs the same model twice on the same dataset:
- `3 columns`: `Script + Titre + Visuel`
- `4 columns`: `Script + Titre + Visuel + Incrustation`

It saves separate outputs, creates per-run graphs, and then builds direct comparison graphs between the two input variants.

## 1) Imports

In [ ]:
import os
import gc
import json
import re
import subprocess
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from vllm import LLM, SamplingParams


## 2) Runtime and GPU configuration

In [ ]:
CUDA_VISIBLE_DEVICES = "0,1"
TENSOR_PARALLEL_SIZE = 2

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"
os.environ["MKL_THREADING_LAYER"] = "GNU"
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("TENSOR_PARALLEL_SIZE:", TENSOR_PARALLEL_SIZE)


## 3) Paths

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv").exists():
    PROJECT_ROOT = Path("/Users/raresolteanu/Desktop/Gliner-Work.Dauphine")

DATASET_PATH = PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv"
OUTPUT_ROOT = PROJECT_ROOT / "communication_function_outputs" / "qwen72b_dual_input"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LATEX_DIR = PROJECT_ROOT / "latex"
LATEX_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_PATH:", DATASET_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## 4) Model configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"
CHAT_MODE = "hf_auto"
MODEL_SLUG = "qwen_qwen2_5_72b_instruct"

TEMPERATURE = 0.0
MAX_NEW_TOKENS = 260
MAX_MODEL_LEN = 8192
VLLM_GPU_MEMORY_UTILIZATION = 0.90
VLLM_SWAP_SPACE_GB = 16
MODEL_DTYPE = "bfloat16"

SMOKE_TEST_N = 20
FULL_RUN_N = None

INPUT_VARIANTS = {
    "three_cols": ["Script", "Titre", "Visuel"],
    "four_cols": ["Script", "Titre", "Visuel", "Incrustation"],
}

MODEL_KWARGS: dict[str, Any] = {
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "trust_remote_code": True,
    "gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION,
    "dtype": MODEL_DTYPE,
    "max_model_len": int(MAX_MODEL_LEN),
    "swap_space": int(VLLM_SWAP_SPACE_GB),
}

print("MODEL_NAME:", MODEL_NAME)
print("INPUT_VARIANTS:", INPUT_VARIANTS)


## 5) Prompt

In [ ]:
RUBRIC_TEXT = 'You are an expert annotation assistant for French automotive advertising.\n\nYour job is to analyze one ad and score it on three communication dimensions:\n- informativeness\n- expressiveness\n- phatic\n\nYou must annotate carefully and conservatively.\nYour goal is not to be creative.\nYour goal is to produce the most defensible annotation possible from the evidence in the ad.\n\nYou will receive ad text that may combine:\n- script\n- on-screen text\n- title\n- visual description\n\nThese elements may contain both literal information and symbolic or rhetorical cues.\nYou must judge the ad as a whole.\n\n==================================================\nTASK\n==================================================\n\nScore the ad on each dimension from 1 to 5.\n\n1 = almost absent\n2 = weak\n3 = moderate\n4 = strong\n5 = very strong\n\nThe three dimensions are independent.\nAn ad can be high on more than one dimension.\nDo not force the scores to sum to any fixed total.\n\nAfter scoring, choose:\n- dominant_dimension\n- dominant_dimension_score\n- confidence\n- reason\n\nIf the highest score is shared by more than one dimension, dominant_dimension must be "mixed".\n\nReturn strict JSON only.\n\n==================================================\nDIMENSION DEFINITIONS\n==================================================\n\nA. INFORMATIVENESS\n\nDefinition:\nHow much the ad provides factual, concrete, product-related, offer-related, or technically useful information.\n\nThis includes:\n- vehicle specifications\n- features and equipment\n- engine or powertrain information\n- electric or hybrid technology\n- charging, range, battery, consumption\n- safety systems\n- comfort or space features when presented concretely\n- maintenance, guarantee, reliability claims when concrete\n- price\n- discounts\n- financing, leasing, monthly payments\n- trade-in conditions\n- bonus or subsidy information\n- model names, versions, product details\n- explicit comparative or functional claims\n\nHigh informativeness means:\nthe ad gives the viewer usable product or offer information.\n\nExamples of signals that increase informativeness:\n- “à partir de 149€/mois”\n- “hybride rechargeable”\n- “3 ans d’entretien inclus”\n- “36 combinaisons de personnalisation”\n- “bonus écologique”\n- “autonomie”\n- “garantie”\n- “équipement”\n- “consommation”\n- “technologie embarquée” when explained concretely\n\nImportant rule:\nPrice, financing, technical details, and offer conditions are informative even if the ad is also emotional.\n\nB. EXPRESSIVENESS\n\nDefinition:\nHow much the ad relies on emotion, desire, identity, aspiration, style, symbolic value, atmosphere, prestige, seduction, or aesthetic projection.\n\nThis includes:\n- emotional appeal\n- beauty, elegance, sensuality\n- pleasure, passion, freedom, adventure\n- self-image and identity\n- desire and dream\n- luxury and prestige\n- symbolic staging\n- strong aestheticization\n- dramatic mood\n- poetic or evocative language\n- brand mythology\n- visual spectacle used to create attraction rather than explain the product\n\nHigh expressiveness means:\nthe ad primarily tries to make the car or brand desirable, meaningful, aspirational, stylish, moving, or emotionally charged.\n\nExamples of signals that increase expressiveness:\n- strong aesthetic staging\n- poetic slogans\n- scenes of freedom, pleasure, seduction, adventure\n- premium aura\n- identity statements\n- emotional music or tone described in the visual text\n- symbolic transformation or fantasy-like presentation\n- language about emotion, desire, dream, elegance, pleasure\n\nImportant rule:\nAn ad can be expressive even if it contains little concrete information.\n\nC. PHATIC\n\nDefinition:\nHow much the ad creates, maintains, or foregrounds social contact, relational connection, conversational closeness, complicity, or audience bonding.\n\nThis includes:\n- direct address to the viewer\n- rhetorical interaction\n- social or conversational tone\n- familiar, intimate, complicit language\n- greetings, invitations, banter, playful contact\n- communication whose main role is to establish or maintain connection rather than inform or aesthetically seduce\n- emphasis on interpersonal exchange, contact, or togetherness\n- “we are talking to you” energy\n- casual rapport-building discourse\n\nHigh phatic means:\nthe ad is strongly oriented toward establishing or maintaining a relationship with the audience or between people.\n\nExamples of signals that increase phatic:\n- direct second-person address used mainly to connect\n- relational invitation\n- joking conversational exchange\n- “let’s stay in touch” style tone\n- social closeness, shared complicity\n- contact-maintaining phrases\n- warm familiarity\n- dialogue where the relationship itself is central\n\nImportant rule:\nPhatic is not the same as emotional.\nPhatic is about contact, relation, and social connection.\nExpressiveness is about emotion, style, desire, and symbolic appeal.\n\n==================================================\nKEY DISTINCTIONS\n==================================================\n\n1. Informativeness vs Expressiveness\n- Informativeness = concrete useful product/offer content\n- Expressiveness = emotional or aesthetic persuasion\n\nIf the ad tells me what the product is, how it works, what it costs, what it includes, or what technical qualities it has, that pushes informativeness upward.\nIf the ad mainly makes me feel something or desire something, that pushes expressiveness upward.\n\n2. Expressiveness vs Phatic\n- Expressiveness = emotion, aspiration, style, symbolism, mood\n- Phatic = contact, social bond, conversational connection, relational presence\n\nA beautiful, emotional, cinematic ad may be highly expressive but not phatic.\nA conversational, joking, audience-facing ad may be phatic without being highly expressive.\n\n3. Informativeness vs Phatic\n- Informativeness gives concrete content\n- Phatic creates contact\n\nA direct-address promotional ad can be both informative and phatic, but only score phatic highly if the relational or contact function is genuinely strong.\n\n==================================================\nVERY IMPORTANT SCORING RULES\n==================================================\n\nUse the full scale carefully.\n\nScore 1:\n- the dimension is nearly absent\n\nScore 2:\n- the dimension is present but weak\n\nScore 3:\n- the dimension is clearly present but not dominant or not strongly emphasized\n\nScore 4:\n- the dimension is strongly present\n\nScore 5:\n- the dimension is central and unmistakably one of the ad’s main communicative forces\n\nDo not inflate all dimensions.\nDo not reward every nice or polished ad with high expressiveness.\nDo not reward every second-person or viewer-facing phrase with high phatic.\nDo not ignore concrete offer or product information.\n\nIf the ad combines a strong emotional frame with many concrete offer details, it may be high in both informativeness and expressiveness.\nIf the ad includes dialogue but the main function is still product or offer explanation, do not automatically raise phatic too much.\n\n==================================================\nSPECIAL AUTOMOTIVE RULES\n==================================================\n\nIn French car ads, the following usually increase informativeness:\n- monthly payment\n- leasing conditions\n- trade-in offers\n- ecological bonus\n- hybrid / electric / rechargeable wording\n- battery / charging / autonomy / range\n- horsepower, engine, consumption\n- guarantee, maintenance, equipment\n- product version, trim, pack, included options\n\nThe following usually increase expressiveness:\n- prestige, elegance, luxury, emotion, freedom\n- cinematic visual spectacle\n- desire, seduction, beauty\n- strong symbolic imagery\n- identity and lifestyle framing\n\nThe following usually increase phatic:\n- conversational closeness\n- relational humor\n- direct interaction whose purpose is social connection\n- “you and us” style bonding\n- complicity or familiarity\n\nImportant:\nHumor alone does not automatically mean phatic.\nEmotion alone does not automatically mean phatic.\nDirect address alone does not automatically mean phatic.\nTechnology alone does not automatically mean informativeness unless it is presented concretely.\n\n==================================================\nDECISION PROCEDURE\n==================================================\n\nFollow this exact reasoning order internally:\n\nStep 1.\nIdentify the ad’s primary communicative force:\n- mainly informing?\n- mainly creating desire/style/emotion?\n- mainly creating social/relational contact?\n- or genuinely mixed?\n\nStep 2.\nIdentify the strongest concrete evidence for each dimension.\n\nStep 3.\nAssign the three scores independently.\n\nStep 4.\nDetermine the dominant dimension from the highest score.\n- If tied at the highest score, dominant_dimension = "mixed"\n\nStep 5.\nSet dominant_dimension_score equal to the highest score.\n\nStep 6.\nGive a very short reason based only on actual evidence from the ad.\n\n==================================================\nOUTPUT REQUIREMENTS\n==================================================\n\nReturn strict JSON only.\n\nUse exactly this schema:\n\n{\n  "informativeness": 1,\n  "expressiveness": 1,\n  "phatic": 1,\n  "dominant_dimension": "informativeness|expressiveness|phatic|mixed",\n  "dominant_dimension_score": 1,\n  "confidence": 0.0,\n  "reason": "short explanation"\n}\n\nConfidence rules:\n- 0.0 to 0.3 = highly uncertain / ambiguous\n- 0.4 to 0.6 = moderately uncertain\n- 0.7 to 0.8 = fairly confident\n- 0.9 to 1.0 = very confident\n\nReason rules:\n- keep it short\n- mention only the strongest evidence\n- do not restate the whole ad\n- do not add extra fields\n- do not add prose outside the JSON\n\n==================================================\nAD TO SCORE\n==================================================\n\n{AD_TEXT}\n\nReturn only strict JSON.'

def output_schema_text() -> str:
    return '''{
  "informativeness": 1,
  "expressiveness": 1,
  "phatic": 1,
  "dominant_dimension": "informativeness|expressiveness|phatic|mixed",
  "dominant_dimension_score": 1,
  "confidence": 0.0,
  "reason": "short explanation"
}'''


## 6) Helpers

In [ ]:
def html_box(title: str, body: str, color: str = "#1f4e79", bg: str = "#eef6fb"):
    display(HTML(
        f"""
        <div style="border-left: 6px solid {color}; background:{bg}; padding:10px 14px; margin:8px 0; border-radius:6px;">
            <div style="font-weight:700; margin-bottom:4px;">{title}</div>
            <div style="white-space:pre-wrap;">{body}</div>
        </div>
        """
    ))

def clean_text(value) -> str:
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def join_labeled_parts(parts: list[tuple[str, str]]) -> str:
    kept = []
    for label, text in parts:
        text = clean_text(text)
        if text:
            kept.append(f"{label}: {text}")
    return "\n".join(kept).strip()

def build_variant_text(row: pd.Series, columns: list[str]) -> str:
    parts = [(col, row.get(col, "")) for col in columns]
    return join_labeled_parts(parts)

def build_prompt_content(ad_text: str) -> str:
    clean_ad_text = " ".join(str(ad_text).split()).strip()
    return RUBRIC_TEXT.replace("{AD_TEXT}", clean_ad_text)

def render_prompt_for_model(content: str, llm=None) -> str:
    if CHAT_MODE == "plain" or llm is None:
        return content
    tokenizer = llm.get_tokenizer()
    messages = [{"role": "user", "content": content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_prompt(ad_text: str, llm=None) -> str:
    return render_prompt_for_model(build_prompt_content(ad_text), llm=llm)

def clamp_score(value, default=1):
    try:
        score = int(round(float(value)))
    except Exception:
        score = default
    return max(1, min(5, score))

def normalize_dimension_name(value):
    text = str(value or "").strip().lower()
    aliases = {{
        "informative": "informativeness",
        "information": "informativeness",
        "referential": "informativeness",
        "expressive": "expressiveness",
        "emotive": "expressiveness",
        "emotion": "expressiveness",
        "phatique": "phatic",
    }}
    text = aliases.get(text, text)
    return text if text in ["informativeness", "expressiveness", "phatic", "mixed"] else ""

def extract_json_object(text: str) -> dict:
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    candidates = [fenced.group(1)] if fenced else []
    candidates.append(text)
    decoder = json.JSONDecoder()
    for candidate in candidates:
        for match in re.finditer(r"\{", candidate):
            try:
                payload, _ = decoder.raw_decode(candidate[match.start():].strip())
                if isinstance(payload, dict):
                    return payload
            except json.JSONDecodeError:
                continue
    raise ValueError("Could not extract JSON from model output.")

def parse_model_prediction(raw_text: str):
    try:
        payload = extract_json_object(raw_text)
        scores = {
            "informativeness": clamp_score(payload.get("informativeness", 1)),
            "expressiveness": clamp_score(payload.get("expressiveness", 1)),
            "phatic": clamp_score(payload.get("phatic", 1)),
        }
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top_score = ranked[0][1]
        top_labels = [k for k, v in ranked if v == top_score]
        dominant_dimension = normalize_dimension_name(payload.get("dominant_dimension")) or ("mixed" if len(top_labels) > 1 else top_labels[0])
        reason = " ".join(str(payload.get("reason", "")).split()).strip() or "strict_json"
        try:
            confidence = round(float(payload.get("confidence", 0.5)), 4)
        except Exception:
            confidence = 0.5
        confidence = max(0.0, min(1.0, confidence))
        return {
            "informativeness": scores["informativeness"],
            "expressiveness": scores["expressiveness"],
            "phatic": scores["phatic"],
            "dominant_dimension": dominant_dimension,
            "dominant_dimension_score": top_score,
            "confidence": confidence,
            "reason": reason,
        }, True, ""
    except Exception as exc:
        return {
            "informativeness": 1,
            "expressiveness": 1,
            "phatic": 1,
            "dominant_dimension": "mixed",
            "dominant_dimension_score": 1,
            "confidence": 0.0,
            "reason": "parse_fallback",
        }, False, str(exc)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_NEW_TOKENS,
)


## 7) Load dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
required_cols = ["row_id", "year", "Marque", "Electric", "Hybrid", "Script", "Titre", "Visuel", "Incrustation"]
required_cols = [c for c in required_cols if c in df.columns]
base_df = df.loc[:, required_cols].copy()

for col in [c for c in ["Script", "Titre", "Visuel", "Incrustation"] if c in base_df.columns]:
    base_df[col] = base_df[col].map(clean_text)

base_df["three_cols_text"] = base_df.apply(lambda row: build_variant_text(row, INPUT_VARIANTS["three_cols"]), axis=1)
base_df["four_cols_text"] = base_df.apply(lambda row: build_variant_text(row, INPUT_VARIANTS["four_cols"]), axis=1)
base_df = base_df[(base_df["three_cols_text"].str.strip() != "") & (base_df["four_cols_text"].str.strip() != "")].copy()

if FULL_RUN_N is not None:
    base_df = base_df.head(FULL_RUN_N).copy()

print("Rows available for both variants:", len(base_df))
display(base_df[["row_id", "three_cols_text", "four_cols_text"]].head(3))


## 8) Variant column check before annotation

In [ ]:
column_overview_df = pd.DataFrame([
    {"variant": "three_cols", "columns_used": ", ".join(INPUT_VARIANTS["three_cols"])},
    {"variant": "four_cols", "columns_used": ", ".join(INPUT_VARIANTS["four_cols"])},
])
html_box("Columns used by the model before annotation", column_overview_df.to_string(index=False))
display(column_overview_df)


## 9) GPU inspection

In [ ]:
def inspect_gpu_state():
    try:
        import torch
        rows = []
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                free_bytes, total_bytes = torch.cuda.mem_get_info(i)
                used_gb = (total_bytes - free_bytes) / (1024**3)
                total_gb = total_bytes / (1024**3)
                rows.append({
                    "gpu_index": i,
                    "name": torch.cuda.get_device_name(i),
                    "used_gb": round(used_gb, 2),
                    "free_gb": round(free_bytes / (1024**3), 2),
                    "total_gb": round(total_gb, 2),
                    "used_percent": round(used_gb / total_gb * 100, 2),
                })
        display(pd.DataFrame(rows))
    except Exception as exc:
        print("Torch GPU inspection failed:", exc)

    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True
        )
        print(result.stdout)
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)


## 10) Inference functions

In [ ]:
def run_variant_batch(sample: pd.DataFrame, text_col: str, variant_name: str, show_prompt_preview: bool = True):
    llm = None
    rows = []
    try:
        llm = LLM(model=MODEL_NAME, **MODEL_KWARGS)
        html_box(f"GPU state after model load: {variant_name}", "The table below shows current GPU memory usage.")
        inspect_gpu_state()

        prompts = [build_prompt(v, llm=llm) for v in sample[text_col].tolist()]
        if show_prompt_preview and prompts:
            print(prompts[0][:900])

        t0 = time.perf_counter()
        try:
            outs = llm.generate(prompts, sampling_params, use_tqdm=True)
        except TypeError:
            outs = llm.generate(prompts, sampling_params)
        t1 = time.perf_counter()

        elapsed = max(0.0, t1 - t0)
        rps = len(sample) / elapsed if elapsed > 0 else float("inf")

        for in_row, out in zip(sample.itertuples(index=False), outs):
            raw_text = out.outputs[0].text if out.outputs else ""
            prediction, parse_ok, parse_error = parse_model_prediction(raw_text)
            rows.append({
                "row_id": int(getattr(in_row, "row_id")),
                "variant_name": variant_name,
                "model_input_text": getattr(in_row, text_col, ""),
                "model_name": MODEL_NAME,
                "raw_output": raw_text,
                "prediction": prediction,
                "parse_ok": parse_ok,
                "parse_error": parse_error,
            })
        return rows, elapsed, rps
    finally:
        try:
            if llm is not None:
                del llm
            gc.collect()
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception:
            pass

def flatten_results(all_results: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    results_df = pd.DataFrame(all_results)
    scores_df = pd.DataFrame([
        {
            "row_id": item.get("row_id"),
            "variant_name": item.get("variant_name", ""),
            "model_input_text": item.get("model_input_text", ""),
            "informativeness": (item.get("prediction") or {}).get("informativeness"),
            "expressiveness": (item.get("prediction") or {}).get("expressiveness"),
            "phatic": (item.get("prediction") or {}).get("phatic"),
            "dominant_dimension": (item.get("prediction") or {}).get("dominant_dimension"),
            "dominant_dimension_score": (item.get("prediction") or {}).get("dominant_dimension_score"),
            "confidence": (item.get("prediction") or {}).get("confidence"),
            "reason": (item.get("prediction") or {}).get("reason"),
            "model_name": item.get("model_name", ""),
            "parse_ok": item.get("parse_ok", False),
        }
        for item in all_results
    ])
    return results_df, scores_df

def run_variant_smoke_and_full(sample_df: pd.DataFrame, text_col: str, variant_name: str):
    smoke_df = sample_df.head(20).copy()
    html_box(f"Smoke test: {variant_name}", f"Running a 20-row smoke test for {variant_name}.")
    smoke_results, smoke_seconds, smoke_rps = run_variant_batch(smoke_df, text_col, variant_name, show_prompt_preview=True)
    smoke_results_df, smoke_scores_df = flatten_results(smoke_results)
    smoke_parse_ok = int(smoke_scores_df["parse_ok"].sum())
    smoke_parse_rate = smoke_parse_ok / len(smoke_scores_df) * 100 if len(smoke_scores_df) else 0.0
    html_box(
        f"Smoke test summary: {variant_name}",
        f"Rows: {len(smoke_scores_df)}\nRows/s: {smoke_rps:.2f}\nParse OK: {smoke_parse_ok}/{len(smoke_scores_df)} ({smoke_parse_rate:.1f}%)"
    )
    display(smoke_scores_df.head(10))

    html_box(f"Full run: {variant_name}", f"Running full annotation for {variant_name} on {len(sample_df)} rows.")
    full_results, full_seconds, full_rps = run_variant_batch(sample_df, text_col, variant_name, show_prompt_preview=False)
    results_df, scores_df = flatten_results(full_results)
    n = len(scores_df)
    parse_ok_count = int(scores_df["parse_ok"].sum())
    parse_ok_rate = parse_ok_count / n * 100 if n else 0.0
    html_box(
        f"Performance summary: {variant_name}",
        f"Rows: {n}\nTotal seconds: {full_seconds:.2f}\nRows per second: {full_rps:.2f}\nParse OK rate: {parse_ok_rate:.2f}%"
    )
    return {
        "variant_name": variant_name,
        "results_df": results_df,
        "scores_df": scores_df,
        "elapsed_seconds": full_seconds,
        "rows_per_second": full_rps,
    }


## 11) Run both variants

In [ ]:
three_run = run_variant_smoke_and_full(base_df, "three_cols_text", "three_cols")
four_run = run_variant_smoke_and_full(base_df, "four_cols_text", "four_cols")

three_scores_df = three_run["scores_df"].copy()
four_scores_df = four_run["scores_df"].copy()


## 12) Save outputs

In [ ]:
def save_variant_outputs(run_obj: dict):
    variant = run_obj["variant_name"]
    results_df = run_obj["results_df"]
    scores_df = run_obj["scores_df"]
    n = len(scores_df)
    variant_dir = OUTPUT_ROOT / variant
    variant_dir.mkdir(parents=True, exist_ok=True)

    jsonl_path = variant_dir / f"{MODEL_SLUG}__{variant}__predictions_full_{n}.jsonl"
    csv_full_path = variant_dir / f"{MODEL_SLUG}__{variant}__predictions_full_{n}_full.csv"
    csv_scores_path = variant_dir / f"{MODEL_SLUG}__{variant}__predictions_full_{n}_scores.csv"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for item in results_df.to_dict(orient="records"):
            record = item.copy()
            if isinstance(record.get("prediction"), dict):
                record["prediction"] = record["prediction"]
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    flat_df = results_df.copy()
    flat_df["prediction"] = flat_df["prediction"].apply(lambda x: json.dumps(x, ensure_ascii=False))
    flat_df.to_csv(csv_full_path, index=False)
    scores_df.to_csv(csv_scores_path, index=False)

    perf_df = pd.DataFrame([{
        "model_name": MODEL_NAME,
        "variant_name": variant,
        "rows": n,
        "rows_per_second": round(run_obj["rows_per_second"], 3),
        "parse_ok_rate_percent": round(scores_df["parse_ok"].mean() * 100, 2),
        "temperature": TEMPERATURE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_model_len": MAX_MODEL_LEN,
        "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
        "gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION,
        "columns_used": ", ".join(INPUT_VARIANTS[variant]),
    }])
    perf_path = variant_dir / f"{MODEL_SLUG}__{variant}__run_summary.csv"
    perf_df.to_csv(perf_path, index=False)
    return {"jsonl_path": jsonl_path, "csv_full_path": csv_full_path, "csv_scores_path": csv_scores_path, "perf_path": perf_path}

three_paths = save_variant_outputs(three_run)
four_paths = save_variant_outputs(four_run)

html_box(
    "Saved outputs",
    f"three_cols scores: {three_paths['csv_scores_path']}\n"
    f"four_cols scores: {four_paths['csv_scores_path']}"
)


## 13) Sanity check

In [ ]:
def sanity_box(scores_df: pd.DataFrame, variant_name: str):
    annotated_rows = len(scores_df)
    unique_row_ids = scores_df["row_id"].nunique()
    missing_score_rows = int(scores_df[["informativeness", "expressiveness", "phatic"]].isna().any(axis=1).sum())
    parse_ok_rows = int(scores_df["parse_ok"].fillna(False).sum())
    html_box(
        f"Sanity check: {variant_name}",
        f"Rows annotated: {annotated_rows}\nUnique row_id: {unique_row_ids}\nRows with missing score values: {missing_score_rows}\nRows parsed successfully: {parse_ok_rows}"
    )

sanity_box(three_scores_df, "three_cols")
sanity_box(four_scores_df, "four_cols")


## 14) Merge metadata for plotting

In [ ]:
meta_cols = [c for c in ["row_id", "year", "Marque", "Electric", "Hybrid", "Visuel"] if c in base_df.columns]
meta_df = base_df[meta_cols].copy()

three_plot_df = three_scores_df.merge(meta_df, on="row_id", how="left")
four_plot_df = four_scores_df.merge(meta_df, on="row_id", how="left")

for df_plot in [three_plot_df, four_plot_df]:
    for col in ["informativeness", "expressiveness", "phatic", "confidence", "dominant_dimension_score"]:
        if col in df_plot.columns:
            df_plot[col] = pd.to_numeric(df_plot[col], errors="coerce")
    if "year" in df_plot.columns:
        df_plot["year"] = pd.to_numeric(df_plot["year"], errors="coerce")
    if "Electric" in df_plot.columns:
        df_plot["Electric"] = pd.to_numeric(df_plot["Electric"], errors="coerce").fillna(0)
    if "Hybrid" in df_plot.columns:
        df_plot["Hybrid"] = pd.to_numeric(df_plot["Hybrid"], errors="coerce").fillna(0)
    if "Electric" in df_plot.columns and "Hybrid" in df_plot.columns:
        df_plot["is_electrified"] = (df_plot["Electric"] > 0) | (df_plot["Hybrid"] > 0)


## 15) Per-variant graph helper

In [ ]:
def save_plot(fig, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved:", path)
    plt.tight_layout()
    plt.show()

def build_variant_graphs(plot_df: pd.DataFrame, variant_name: str):
    score_cols = ["informativeness", "expressiveness", "phatic"]
    plot_dir = LATEX_DIR / variant_name
    plot_dir.mkdir(parents=True, exist_ok=True)

    # 1. score histograms
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    bins = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
    for ax, col in zip(axes, score_cols):
        ax.hist(plot_df[col].dropna(), bins=bins, edgecolor="black", alpha=0.8, color="#C08081")
        ax.set_title(f"{variant_name} - {col}")
        ax.set_xticks([1,2,3,4,5])
        ax.grid(axis="y", alpha=0.25)
    axes[0].set_ylabel("Count")
    save_plot(fig, plot_dir / "01_score_histograms.png")

    # 2. correlation heatmap
    corr_df = plot_df[score_cols].corr(method="pearson")
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr_df.values, cmap="coolwarm", vmin=-1, vmax=1)
    fig.colorbar(im, ax=ax)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(score_cols, rotation=20, ha="right")
    ax.set_yticklabels(score_cols)
    ax.set_title(f"{variant_name} correlation")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{corr_df.values[i,j]:.2f}", ha="center", va="center")
    save_plot(fig, plot_dir / "02_correlation_heatmap.png")

    # 3. yearly trends
    if "year" in plot_df.columns and plot_df["year"].notna().any():
        tmp = plot_df.dropna(subset=["year"]).copy()
        tmp["year"] = tmp["year"].astype(int)
        yearly = tmp.groupby("year", as_index=False)[score_cols].mean()
        fig, ax = plt.subplots(figsize=(11, 5))
        colors = {"informativeness": "#C08081", "expressiveness": "#5B8E7D", "phatic": "#C7A64A"}
        for col in score_cols:
            ax.plot(yearly["year"], yearly[col], marker="o", linewidth=2.0, label=col, color=colors[col])
        ax.set_title(f"{variant_name} yearly trends")
        ax.set_ylim(1, 5)
        ax.grid(axis="y", alpha=0.25)
        ax.legend()
        save_plot(fig, plot_dir / "03_yearly_trends.png")

    # 4. top brands
    if "Marque" in plot_df.columns:
        tmp = plot_df.dropna(subset=["Marque"]).copy()
        top_brands = tmp["Marque"].astype(str).value_counts().head(10).index.tolist()
        tmp = tmp[tmp["Marque"].astype(str).isin(top_brands)].copy()
        brand_means = tmp.groupby("Marque", as_index=False)[score_cols].mean()
        brand_means["order"] = pd.Categorical(brand_means["Marque"], categories=top_brands, ordered=True)
        brand_means = brand_means.sort_values("order")
        fig, ax = plt.subplots(figsize=(13, 7))
        y = np.arange(len(brand_means))
        offsets = {"informativeness": -0.24, "expressiveness": 0.0, "phatic": 0.24}
        colors = {"informativeness": "#C08081", "expressiveness": "#5B8E7D", "phatic": "#C7A64A"}
        for col in score_cols:
            ax.barh(y + offsets[col], brand_means[col], height=0.22, label=col, color=colors[col])
        ax.set_yticks(y)
        ax.set_yticklabels(brand_means["Marque"])
        ax.set_xlim(1, 5)
        ax.set_title(f"{variant_name} top brands")
        ax.legend()
        ax.grid(axis="x", alpha=0.25)
        ax.invert_yaxis()
        save_plot(fig, plot_dir / "04_top_brands.png")

    # 5. EV vs non-EV
    if "is_electrified" in plot_df.columns:
        ev_means = plot_df.groupby("is_electrified", as_index=False)[score_cols].mean()
        ev_means["segment"] = ev_means["is_electrified"].map({False: "Non electrified", True: "Electrified"})
        fig, ax = plt.subplots(figsize=(10, 5))
        y = np.arange(len(ev_means))
        offsets = {"informativeness": -0.24, "expressiveness": 0.0, "phatic": 0.24}
        colors = {"informativeness": "#C08081", "expressiveness": "#5B8E7D", "phatic": "#C7A64A"}
        for col in score_cols:
            ax.barh(y + offsets[col], ev_means[col], height=0.22, label=col, color=colors[col])
        ax.set_yticks(y)
        ax.set_yticklabels(ev_means["segment"])
        ax.set_xlim(1, 5)
        ax.set_title(f"{variant_name} EV vs non-EV")
        ax.legend()
        ax.grid(axis="x", alpha=0.25)
        ax.invert_yaxis()
        save_plot(fig, plot_dir / "05_ev_split.png")

    # 6. dominant share
    dom = plot_df["dominant_dimension"].value_counts(normalize=True).mul(100).reindex(["informativeness", "expressiveness", "phatic", "mixed"], fill_value=0)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(dom.index, dom.values, color=["#C08081", "#5B8E7D", "#C7A64A", "#7F7F7F"])
    ax.set_title(f"{variant_name} dominant-dimension shares")
    ax.set_ylabel("Percent")
    ax.grid(axis="y", alpha=0.25)
    save_plot(fig, plot_dir / "06_dominant_shares.png")

    # 7. confidence histogram
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(plot_df["confidence"].dropna(), bins=10, edgecolor="black", color="#5B8E7D", alpha=0.8)
    ax.set_title(f"{variant_name} confidence distribution")
    ax.grid(axis="y", alpha=0.25)
    save_plot(fig, plot_dir / "07_confidence_hist.png")

    # 8. dominant score histogram
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(plot_df["dominant_dimension_score"].dropna(), bins=[0.5,1.5,2.5,3.5,4.5,5.5], edgecolor="black", color="#C7A64A", alpha=0.8)
    ax.set_title(f"{variant_name} dominant score distribution")
    ax.set_xticks([1,2,3,4,5])
    ax.grid(axis="y", alpha=0.25)
    save_plot(fig, plot_dir / "08_dominant_score_hist.png")

    # 9. parse-ok pie
    parse_counts = plot_df["parse_ok"].fillna(False).value_counts()
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(parse_counts.values, labels=[str(x) for x in parse_counts.index], autopct="%1.1f%%", colors=["#5B8E7D", "#C08081"])
    ax.set_title(f"{variant_name} parse success")
    save_plot(fig, plot_dir / "09_parse_ok_pie.png")

    # 10. score boxplots
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot([plot_df[c].dropna() for c in score_cols], labels=score_cols)
    ax.set_title(f"{variant_name} score boxplots")
    ax.set_ylim(1, 5)
    ax.grid(axis="y", alpha=0.25)
    save_plot(fig, plot_dir / "10_score_boxplots.png")

    # 11. rowwise total score histogram
    fig, ax = plt.subplots(figsize=(7, 4))
    total_score = plot_df[score_cols].sum(axis=1)
    ax.hist(total_score.dropna(), bins=12, edgecolor="black", color="#7F7F7F", alpha=0.8)
    ax.set_title(f"{variant_name} total score distribution")
    ax.grid(axis="y", alpha=0.25)
    save_plot(fig, plot_dir / "11_total_score_hist.png")

    # 12. score means with error bars
    means = plot_df[score_cols].mean()
    stds = plot_df[score_cols].std()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(score_cols, means.values, yerr=stds.values, color=["#C08081", "#5B8E7D", "#C7A64A"], capsize=5)
    ax.set_ylim(1, 5)
    ax.set_title(f"{variant_name} means and variability")
    ax.grid(axis="y", alpha=0.25)
    save_plot(fig, plot_dir / "12_means_with_errorbars.png")

    # 13. duplicate visuel diagnostics
    if "Visuel" in plot_df.columns:
        tmp = plot_df.copy()
        tmp["Visuel"] = tmp["Visuel"].astype(str).str.strip()
        tmp["dup_count"] = tmp.groupby("Visuel")["row_id"].transform("size")
        tmp["dup_bucket"] = tmp["dup_count"].map(lambda n: "1" if n == 1 else ("2-5" if n <= 5 else ("6-10" if n <= 10 else "11+")))
        dup_means = tmp.groupby("dup_bucket", as_index=False)[score_cols].mean()
        order = ["1", "2-5", "6-10", "11+"]
        dup_means["dup_bucket"] = pd.Categorical(dup_means["dup_bucket"], categories=order, ordered=True)
        dup_means = dup_means.sort_values("dup_bucket")
        fig, ax = plt.subplots(figsize=(10, 5))
        for col, color in [("informativeness", "#C08081"), ("expressiveness", "#5B8E7D"), ("phatic", "#C7A64A")]:
            ax.plot(dup_means["dup_bucket"].astype(str), dup_means[col], marker="o", linewidth=2.0, label=col, color=color)
        ax.set_ylim(1, 5)
        ax.set_title(f"{variant_name} duplicate creative diagnostics")
        ax.grid(axis="y", alpha=0.25)
        ax.legend()
        save_plot(fig, plot_dir / "13_duplicate_diagnostics.png")

    # 14. year counts
    if "year" in plot_df.columns and plot_df["year"].notna().any():
        tmp = plot_df.dropna(subset=["year"]).copy()
        tmp["year"] = tmp["year"].astype(int)
        year_counts = tmp["year"].value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(year_counts.index.astype(str), year_counts.values, color="#7F7F7F")
        ax.set_title(f"{variant_name} rows per year")
        ax.grid(axis="y", alpha=0.25)
        plt.xticks(rotation=30, ha="right")
        save_plot(fig, plot_dir / "14_rows_per_year.png")

    # 15. brand count distribution top 15
    if "Marque" in plot_df.columns:
        brand_counts = plot_df["Marque"].astype(str).value_counts().head(15)
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.bar(brand_counts.index, brand_counts.values, color="#5B8E7D")
        ax.set_title(f"{variant_name} top 15 brand frequencies")
        ax.grid(axis="y", alpha=0.25)
        plt.xticks(rotation=35, ha="right")
        save_plot(fig, plot_dir / "15_top15_brand_counts.png")


## 16) Build 15 graphs for the 3-column run

In [ ]:
build_variant_graphs(three_plot_df, "three_cols")

## 17) Build 15 graphs for the 4-column run

In [ ]:
build_variant_graphs(four_plot_df, "four_cols")

## 18) Build 5 direct comparison graphs between the two variants

In [ ]:
comparison_df = three_scores_df.merge(
    four_scores_df,
    on="row_id",
    suffixes=("_3col", "_4col")
)

# 1. mean scores comparison
fig, ax = plt.subplots(figsize=(9, 5))
mean_df = pd.DataFrame({
    "dimension": ["informativeness", "expressiveness", "phatic"],
    "three_cols": [three_scores_df["informativeness"].mean(), three_scores_df["expressiveness"].mean(), three_scores_df["phatic"].mean()],
    "four_cols": [four_scores_df["informativeness"].mean(), four_scores_df["expressiveness"].mean(), four_scores_df["phatic"].mean()],
})
x = np.arange(len(mean_df))
ax.bar(x - 0.18, mean_df["three_cols"], 0.36, label="three_cols", color="#C08081")
ax.bar(x + 0.18, mean_df["four_cols"], 0.36, label="four_cols", color="#5B8E7D")
ax.set_xticks(x)
ax.set_xticklabels(mean_df["dimension"])
ax.set_ylim(1, 5)
ax.set_title("Comparison 1: mean scores")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_plot(fig, LATEX_DIR / "comparison_01_mean_scores.png")

# 2. dominant-dimension share comparison
dom3 = three_scores_df["dominant_dimension"].value_counts(normalize=True).mul(100).reindex(["informativeness","expressiveness","phatic","mixed"], fill_value=0)
dom4 = four_scores_df["dominant_dimension"].value_counts(normalize=True).mul(100).reindex(["informativeness","expressiveness","phatic","mixed"], fill_value=0)
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(4)
ax.bar(x - 0.18, dom3.values, 0.36, label="three_cols", color="#C08081")
ax.bar(x + 0.18, dom4.values, 0.36, label="four_cols", color="#5B8E7D")
ax.set_xticks(x)
ax.set_xticklabels(dom3.index)
ax.set_title("Comparison 2: dominant-dimension shares")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_plot(fig, LATEX_DIR / "comparison_02_dominant_shares.png")

# 3. exact and within-1-point agreement between the two variants
agree_rows = []
for col in ["informativeness", "expressiveness", "phatic"]:
    exact = (comparison_df[f"{col}_3col"] == comparison_df[f"{col}_4col"]).mean() * 100
    within1 = ((comparison_df[f"{col}_3col"] - comparison_df[f"{col}_4col"]).abs() <= 1).mean() * 100
    agree_rows.append({"dimension": col, "exact": exact, "within1": within1})
agree_df = pd.DataFrame(agree_rows)
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(agree_df))
ax.bar(x - 0.18, agree_df["exact"], 0.36, label="exact", color="#C08081")
ax.bar(x + 0.18, agree_df["within1"], 0.36, label="within 1 point", color="#5B8E7D")
ax.set_xticks(x)
ax.set_xticklabels(agree_df["dimension"])
ax.set_title("Comparison 3: agreement between 3-column and 4-column runs")
ax.set_ylabel("Percent")
ax.legend()
ax.grid(axis="y", alpha=0.25)
save_plot(fig, LATEX_DIR / "comparison_03_agreement.png")

# 4. expressiveness heatmap between the two variants
heat = pd.crosstab(comparison_df["expressiveness_3col"], comparison_df["expressiveness_4col"]).reindex(index=[1,2,3,4,5], columns=[1,2,3,4,5], fill_value=0)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(heat.values, cmap="YlOrRd")
fig.colorbar(im, ax=ax)
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels([1,2,3,4,5])
ax.set_yticklabels([1,2,3,4,5])
ax.set_xlabel("4 columns")
ax.set_ylabel("3 columns")
ax.set_title("Comparison 4: expressiveness heatmap")
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(int(heat.values[i, j])), ha="center", va="center")
save_plot(fig, LATEX_DIR / "comparison_04_expressiveness_heatmap.png")

# 5. yearly trends comparison
three_year = three_plot_df.groupby("year", as_index=False)[["informativeness", "expressiveness", "phatic"]].mean().dropna()
four_year = four_plot_df.groupby("year", as_index=False)[["informativeness", "expressiveness", "phatic"]].mean().dropna()
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, col in zip(axes, ["informativeness", "expressiveness", "phatic"]):
    ax.plot(three_year["year"], three_year[col], marker="o", linewidth=2.0, label="three_cols", color="#C08081")
    ax.plot(four_year["year"], four_year[col], marker="o", linewidth=2.0, label="four_cols", color="#5B8E7D")
    ax.set_title(col)
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Mean score")
axes[-1].legend()
save_plot(fig, LATEX_DIR / "comparison_05_yearly_trends.png")
